# Day 1 — Project Setup + Data Ingestion (ETL)

**Project:** Mutual Fund Analytics Capstone  
**Role:** Data Analyst Intern  
**Objective:** Complete Day 1 ETL work using a Jupyter Notebook.

This notebook performs:
- Project folder setup
- Loading all 10 CSV datasets
- `.shape`, `.dtypes`, `.head()` checks
- Data quality checks
- Fund master exploration
- AMFI code validation
- Live NAV fetching from mfapi.in
- Data quality summary report generation

## Step 1 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path

## Step 2 — Setup Project Folder Paths

In [ ]:
current_path = Path.cwd()

# If notebook is running from notebooks folder, move one level up
if current_path.name.lower() == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
REPORT_PATH = PROJECT_ROOT / "reports"
SQL_PATH = PROJECT_ROOT / "sql"
DASHBOARD_PATH = PROJECT_ROOT / "dashboard"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks"

for folder in [RAW_PATH, PROCESSED_PATH, REPORT_PATH, SQL_PATH, DASHBOARD_PATH, NOTEBOOK_PATH]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw Path:", RAW_PATH)
print("Processed Path:", PROCESSED_PATH)
print("Reports Path:", REPORT_PATH)

## Step 3 — Check Required CSV Files

In [ ]:
files = [
    "01_fund_master.csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

for file in files:
    path = RAW_PATH / file
    print(file, "FOUND" if path.exists() else "MISSING")

## Step 4 — Load All 10 CSV Files Using Pandas

In [ ]:
datasets = {}

for file in files:
    path = RAW_PATH / file
    df = pd.read_csv(path)
    datasets[file.replace(".csv", "")] = df

    print("\n" + "=" * 80)
    print("File:", file)
    print("Shape:", df.shape)
    print("\nData Types:")
    print(df.dtypes)
    print("\nFirst 5 Rows:")
    display(df.head())

## Step 5 — Create Dataset Summary

In [ ]:
summary_df = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": int(df.isnull().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    }
    for name, df in datasets.items()
])

summary_df

## Step 6 — Check Missing Values

In [ ]:
missing_report = []

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    for col, count in missing.items():
        missing_report.append({
            "dataset": name,
            "column": col,
            "missing_count": int(count)
        })

missing_df = pd.DataFrame(missing_report)

if missing_df.empty:
    print("No missing values found in initial check.")
else:
    display(missing_df)

## Step 7 — Explore Fund Master Dataset

In [ ]:
fund_master = datasets["01_fund_master"]

print("Unique Fund Houses:")
print(fund_master["fund_house"].dropna().unique())

print("\nUnique Categories:")
print(fund_master["category"].dropna().unique())

print("\nUnique Sub-Categories:")
print(fund_master["sub_category"].dropna().unique())

print("\nUnique Risk Grades:")
print(fund_master["risk_category"].dropna().unique())

print("\nAMFI Code Sample:")
display(fund_master[["amfi_code", "fund_house", "scheme_name", "category", "sub_category", "risk_category"]].head(10))

## Step 8 — Understand AMFI Code Structure

`amfi_code` is a unique numeric identifier for each mutual fund scheme.  
In this project, `amfi_code` is used as a key to connect fund master data with NAV history and other scheme-level datasets.

## Step 9 — Validate AMFI Codes

In [ ]:
nav_history = datasets["02_nav_history"]

master_codes = set(fund_master["amfi_code"].dropna().astype(str))
nav_codes = set(nav_history["amfi_code"].dropna().astype(str))

missing_codes = sorted(master_codes - nav_codes)
extra_codes = sorted(nav_codes - master_codes)

print("Total AMFI codes in fund_master:", len(master_codes))
print("Total AMFI codes in nav_history:", len(nav_codes))
print("Codes missing in nav_history:", missing_codes)
print("Extra codes in nav_history:", extra_codes[:20])

if len(missing_codes) == 0:
    validation_result = "PASS - Every AMFI code in fund_master exists in nav_history."
else:
    validation_result = "FAIL - Some AMFI codes are missing from nav_history."

print("\nValidation Result:", validation_result)

## Step 10 — Fetch Live NAV Data from mfapi.in

In [ ]:
scheme_codes = {
    "HDFC Top 100 Direct": 125497,
    "SBI Bluechip": 119551,
    "ICICI Bluechip": 120503,
    "Nippon Large Cap": 118632,
    "Axis Bluechip": 119092,
    "Kotak Bluechip": 120841
}

all_nav_rows = []

for scheme_name, code in scheme_codes.items():
    url = f"https://api.mfapi.in/mf/{code}"
    print("Fetching:", scheme_name, url)

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    json_data = response.json()

    meta = json_data.get("meta", {})
    nav_data = json_data.get("data", [])

    for row in nav_data:
        all_nav_rows.append({
            "input_scheme_name": scheme_name,
            "amfi_code": code,
            "fund_house": meta.get("fund_house"),
            "scheme_type": meta.get("scheme_type"),
            "scheme_category": meta.get("scheme_category"),
            "scheme_code": meta.get("scheme_code"),
            "scheme_name": meta.get("scheme_name"),
            "date": row.get("date"),
            "nav": row.get("nav")
        })

live_nav_df = pd.DataFrame(all_nav_rows)
live_nav_df["date"] = pd.to_datetime(live_nav_df["date"], format="%d-%m-%Y", errors="coerce")
live_nav_df["nav"] = pd.to_numeric(live_nav_df["nav"], errors="coerce")

live_nav_file = RAW_PATH / "live_nav_data.csv"
live_nav_df.to_csv(live_nav_file, index=False)

print("Live NAV saved successfully:", live_nav_file)
print("Shape:", live_nav_df.shape)
display(live_nav_df.head())

## Step 11 — Save Processed Copies

In [ ]:
for name, df in datasets.items():
    output_file = PROCESSED_PATH / f"{name}_processed.csv"
    df.to_csv(output_file, index=False)

print("Processed files saved successfully in:", PROCESSED_PATH)

## Step 12 — Create Data Quality Summary Report

In [ ]:
report_lines = []

report_lines.append("DAY 1 DATA QUALITY SUMMARY")
report_lines.append("=" * 70)

report_lines.append("\n1. Dataset Overview")
report_lines.append(summary_df.to_string(index=False))

report_lines.append("\n\n2. Missing Values / Anomalies")
if missing_df.empty:
    report_lines.append("No major missing values found during initial check.")
else:
    report_lines.append(missing_df.to_string(index=False))

report_lines.append("\n\n3. Fund Master Exploration")
report_lines.append(f"Unique fund houses: {fund_master['fund_house'].nunique()}")
report_lines.append(f"Unique categories: {fund_master['category'].nunique()}")
report_lines.append(f"Unique sub-categories: {fund_master['sub_category'].nunique()}")
report_lines.append(f"Unique risk grades: {fund_master['risk_category'].nunique()}")

report_lines.append("\n\n4. AMFI Validation")
report_lines.append(validation_result)

report_lines.append("\n\n5. Live NAV API Fetch")
report_lines.append(f"Live NAV fetched for {len(scheme_codes)} schemes.")
report_lines.append(f"live_nav_data.csv shape: {live_nav_df.shape}")

summary_file = REPORT_PATH / "day1_data_quality_summary.txt"
summary_file.write_text("\n".join(report_lines), encoding="utf-8")

print("\n".join(report_lines))
print("\nReport saved at:", summary_file)

## Step 13 — Final Day 1 Checklist

In [ ]:
print("DAY 1 COMPLETED")
print("Raw data folder exists:", RAW_PATH.exists())
print("Processed data folder exists:", PROCESSED_PATH.exists())
print("Report folder exists:", REPORT_PATH.exists())
print("Live NAV file exists:", (RAW_PATH / "live_nav_data.csv").exists())
print("Quality report exists:", (REPORT_PATH / "day1_data_quality_summary.txt").exists())

print("\nGit Commit Message:")
print('git commit -m "Day 1: Data ingestion complete"')